In [1]:
import pandas as pd
import glob

Consolidating Fundamentals

In [2]:
symbols = ['GOOGL', 'META', 'MSFT', 'NVDA', 'PLTR']


In [13]:
all_files = []
for symbol in symbols:

    csv_files = glob.glob(f'/workspaces/bubbleindex/bronze/{symbol}/*.csv')
    all_files.append(csv_files)

In [14]:
print(all_files)

[['/workspaces/bubbleindex/bronze/GOOGL/GOOGL_price.csv', '/workspaces/bubbleindex/bronze/GOOGL/GOOGL_cashflow.csv', '/workspaces/bubbleindex/bronze/GOOGL/GOOGL_incomestatement.csv', '/workspaces/bubbleindex/bronze/GOOGL/GOOGL_balancesheet.csv'], ['/workspaces/bubbleindex/bronze/META/META_price.csv', '/workspaces/bubbleindex/bronze/META/META_balancesheet.csv', '/workspaces/bubbleindex/bronze/META/META_incomestatement.csv', '/workspaces/bubbleindex/bronze/META/META_cashflow.csv'], ['/workspaces/bubbleindex/bronze/MSFT/MSFT_price.csv', '/workspaces/bubbleindex/bronze/MSFT/MSFT_balancesheet.csv', '/workspaces/bubbleindex/bronze/MSFT/MSFT_cashflow.csv', '/workspaces/bubbleindex/bronze/MSFT/MSFT_incomestatement.csv'], ['/workspaces/bubbleindex/bronze/NVDA/NVDA_price.csv', '/workspaces/bubbleindex/bronze/NVDA/NVDA_cashflow.csv', '/workspaces/bubbleindex/bronze/NVDA/NVDA_balancesheet.csv', '/workspaces/bubbleindex/bronze/NVDA/NVDA_incomestatement.csv'], ['/workspaces/bubbleindex/bronze/PLTR/P

In [21]:

#list of fundamentals for each company (googldf, metadf, pltrdf2,etc)
final_fundamentals_dfs = []
#symbol = ['/workspaces/bubbleindex/bronze/GOOGL/GOOGL_price.csv', 
#           '/workspaces/bubbleindex/bronze/GOOGL/GOOGL_cashflow.csv', 
#       # '/workspaces/bubbleindex/bronze/GOOGL/GOOGL_incomestatement.csv', 
#       '/workspaces/bubbleindex/bronze/GOOGL/GOOGL_balancesheet.csv']
for symbol in all_files:
    #list of all fundamentals dfs (df,df1,df2)
    fundamentals_files = []

    #csvfile = '/workspaces/bubbleindex/bronze/GOOGL/GOOGL_balancesheet.csv
    for csv_file in symbol:
        #if csv_file is a price file skip it
        if not csv_file.endswith('_price.csv'):
            df = pd.read_csv(csv_file)
            df = df.rename(columns={'Unnamed: 0': 'Date'}) 

            #filter columns based on file type
            if csv_file.endswith('_balancesheet.csv'):
                df = df[['ticker', 'Date', 'Total Debt', 'Working Capital']]
            elif csv_file.endswith('_cashflow.csv'):
                df = df[['ticker', 'Date', 'Free Cash Flow', 'Beginning Cash Position' , 'End Cash Position']]
            else:
                df = df[['ticker', 'Date', 'Total Revenue', 'Gross Profit', 'Net Income', 'Diluted EPS', 'Research And Development', 'EBITDA', 'Tax Provision']]
            fundamentals_files.append(df)
    
    fundamentals_df = fundamentals_files[0].merge(fundamentals_files[1], on=['ticker', 'Date'], how='inner').merge(fundamentals_files[2], on=['ticker', 'Date'], how='inner')
    #fundamentals_df.to_csv(f'/workspaces/bubbleindex/silver/fundamentals.csv', index=False)
    final_fundamentals_dfs.append(fundamentals_df)


# combine all fundamentals dfs into one
combined_fundamentals = pd.concat([final_fundamentals_dfs[0], final_fundamentals_dfs[1], final_fundamentals_dfs[2], final_fundamentals_dfs[3], final_fundamentals_dfs[4]],  ignore_index=True)
combined_fundamentals.to_csv('/workspaces/bubbleindex/silver/combined_fundamentals.csv', index=False)
        

Feature Engineering (Ratios & Growth)

Convert raw values into comparable metrics:

Margins:

Gross Margin = Gross Profit ÷ Total Revenue

Net Margin = Net Income ÷ Total Revenue

FCF Margin = Free Cash Flow ÷ Total Revenue

Leverage & Liquidity:

Debt-to-EBITDA = Total Debt ÷ EBITDA

Working Capital Ratio = Working Capital ÷ Total Revenue

Innovation:

R&D Intensity = Research & Development ÷ Total Revenue

In [ ]:
df['Gross Margin'] = df['Gross Profit'] / df['Total Revenue']
df['Net Margin'] = df['Net Income'] / df['Total Revenue']
df['FCF Margin'] = df['Free Cash Flow'] / df['Total Revenue']

In [ ]:
df['Debt to Equity'] = df['Total Debt'] / (df['Total Debt'] + df['Working Capital'])
df['Working Capital Ratio'] = df['Working Capital'] / df['Total Revenue']
df['R&D to Revenue'] = df['Research And Development'] / df['Total Revenue']

In [3]:
df = pd.read_csv('/workspaces/bubbleindex/silver/combined_fundamentals.csv')
df

,ticker,Date,Free Cash Flow,Beginning Cash Position,End Cash Position,Total Revenue,Gross Profit,Net Income,Diluted EPS,Research And Development,EBITDA,Tax Provision,Total Debt,Working Capital
0,GOOGL,2025-12-31,7.326600e+10,2.346600e+10,3.070800e+10,4.028360e+11,2.403010e+11,1.321700e+11,10.810000,6.108700e+10,1.806980e+11,2.665600e+10,5.929100e+10,1.032930e+11
1,GOOGL,2024-12-31,7.276400e+10,2.404800e+10,2.346600e+10,3.500180e+11,2.037120e+11,1.001180e+11,8.040000,4.932600e+10,1.353940e+11,1.969700e+10,2.257400e+10,7.458900e+10
2,GOOGL,2023-12-31,6.949500e+10,2.187900e+10,2.404800e+10,3.073940e+11,1.740620e+11,7.379500e+10,5.800000,4.542700e+10,9.797100e+10,1.192200e+10,2.712100e+10,8.971600e+10
3,GOOGL,2022-12-31,6.001000e+10,2.094500e+10,2.187900e+10,2.828360e+11,1.566330e+11,5.997200e+10,4.560000,3.950000e+10,8.516000e+10,1.135600e+10,2.967900e+10,9.549500e+10
4,GOOGL,2021-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,META,2025-12-31,4.610900e+10,4.543800e+10,3.910000e+10,2.009660e+11,1.647910e+11,6.045800e+10,NaN,5.737200e+10,1.057130e+11,2.547400e+10,8.389700e+10,6.688600e+10
6,META,2024-12-31,5.407200e+10,4.282700e+10,4.543800e+10,1.645010e+11,1.343400e+11,6.236000e+10,23.860000,4.387300e+10,8.687600e+10,8.303000e+09,4.906000e+10,6.644900e+10
7,META,2023-12-31,4.406800e+10,1.559600e+10,4.282700e+10,1.349020e+11,1.089430e+11,3.909800e+10,14.870000,3.848300e+10,5.905200e+10,8.330000e+09,3.723400e+10,5.340500e+10
8,META,2022-12-31,1.928900e+10,1.686500e+10,1.559600e+10,1.166090e+11,9.136000e+10,2.320000e+10,8.590000,3.533800e+10,3.769000e+10,5.619000e+09,2.659100e+10,3.252300e+10
9,META,2021-12-31,NaN,NaN,NaN,NaN,NaN,NaN,13.770000,NaN,NaN,NaN,NaN,NaN


In [7]:
df['Date'] = pd.to_datetime(df['Date'])

Metrics

Gross Margin = Gross Profit ÷ Total Revenue

In [8]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   ticker                    24 non-null     str           
 1   Date                      24 non-null     datetime64[us]
 2   Free Cash Flow            20 non-null     float64       
 3   Beginning Cash Position   20 non-null     float64       
 4   End Cash Position         20 non-null     float64       
 5   Total Revenue             20 non-null     float64       
 6   Gross Profit              20 non-null     float64       
 7   Net Income                20 non-null     float64       
 8   Diluted EPS               20 non-null     float64       
 9   Research And Development  20 non-null     float64       
 10  EBITDA                    20 non-null     float64       
 11  Tax Provision             20 non-null     float64       
 12  Total Debt                20 non-nu